# CA.02: First Contact
### CSCI 4220 · Coding assignment · Transformers and large language models

Robby's radio telescope picked up a signal. It's arithmetic, in a language nobody on Earth speaks:

```
vek(6, 14) = 20        drom(6, 3) = 18        skal(6, 18) = 12
```

First **you** crack the code. Then you teach a 494-million-parameter transformer (Qwen2.5-0.5B-Instruct)
the same language five different ways, and find out which ones work.

| Part | | Points |
|---|---|---|
| 1 | Crack the code | 10 |
| 2 | Inside the transformer: tokens, probabilities, attention | 15 |
| 3 | Base prompting | 5 |
| 4 | System prompting | 10 |
| 5 | Chain-of-thought | 15 |
| 6 | In-context learning | 10 |
| 7 | LoRA fine-tuning | 25 |
| 8 | Analysis | 10 |
| ★ | Bonus: prompt leaderboard | +5 |

**Before you start:** in Colab, *Runtime → Change runtime type → T4 GPU*. Running everything takes about 25 minutes.

**Submit:** your notebook as a PDF (*File → Print → Save as PDF*) with all outputs visible.
AI tools are permitted and encouraged; if you use one, say so at the top and say what it did for you.

*Inspired by "Alien CalcGPT" (CS 229, Homework 1).*

## Setup
Run these cells once. **Don't edit them.**

In [ ]:
%pip install -q peft
import random, re
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, attn_implementation="eager").to(device)
model.eval()
print(f"Loaded {MODEL_NAME} on {device}")

In [ ]:
# ── the signal: 300 training problems and 90 test problems at three levels ──
OPS = ("vek", "drom", "skal")

def _leaf(rng, op):
    return rng.randint(1, 12 if op == "drom" else 20)

def _make(op, x, y):
    # x, y are (alien, normal math, value); skal never goes negative
    if op == "skal" and y[2] < x[2]:
        x, y = y, x
    value = {"vek": x[2] + y[2], "drom": x[2] * y[2], "skal": y[2] - x[2]}[op]
    w = lambda t: f"({t[1]})" if any(s in t[1] for s in "+-*") else t[1]
    normal = {"vek": f"{w(x)} + {w(y)}", "drom": f"{w(x)} * {w(y)}", "skal": f"{w(y)} - {w(x)}"}[op]
    return f"{op}({x[0]}, {y[0]})", normal, value

def make_problem(level, rng):
    op = rng.choice(OPS)
    num = lambda o: (lambda n: (str(n), str(n), n))(_leaf(rng, o))
    if level == 1:
        return _make(op, num(op), num(op))
    if level == 2:
        i = rng.choice(OPS); inner = _make(i, num(i), num(i))
        return _make(op, inner, num(op)) if rng.random() < 0.5 else _make(op, num(op), inner)
    a, b = rng.choice(OPS), rng.choice(OPS)
    return _make(op, _make(a, num(a), num(a)), _make(b, num(b), num(b)))

def make_split(n_per_level, rng, avoid=()):
    out = ([], [], [], [])
    for level in (1, 2, 3):
        made = 0
        while made < n_per_level:
            p, s, a = make_problem(level, rng)
            if p in avoid or p in out[0]:
                continue
            for lst, v in zip(out, (p, s, a, level)): lst.append(v)
            made += 1
    return out

rng = random.Random(4220)
train_problems, train_normal, train_answers, train_levels = make_split(100, rng)
test_problems, test_normal, test_answers, test_levels = make_split(30, rng, avoid=set(train_problems))
print(f"train: {len(train_problems)}   test: {len(test_problems)}")
print("level 2 looks like:", train_problems[100], "=", train_answers[100])
print("level 3 looks like:", train_problems[200], "=", train_answers[200])

In [ ]:
# ── helpers ──
def generate_response(messages, model=model, max_new_tokens=64):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def extract_answer(response):
    # the integer after "Final Answer"; if there isn't one, the last "= number" in the reply
    m = re.search(r"Final Answer\W*?(-?\d+)", response, re.S)
    if m:
        return int(m.group(1))
    ends = re.findall(r"=\s*(-?\d+)\s*$", response, re.M)
    return int(ends[-1]) if ends else None

results = {}

def evaluate(name, prompt_fn, model=model, max_new_tokens=64, levels=(1, 2, 3)):
    idx = [i for i, lv in enumerate(test_levels) if lv in levels]
    outputs = [generate_response(prompt_fn(test_problems[i]), model, max_new_tokens) for i in tqdm(idx, desc=name)]
    preds = [extract_answer(o) for o in outputs]
    correct = [p == test_answers[i] for p, i in zip(preds, idx)]
    results[name] = {"idx": idx, "preds": preds, "outputs": outputs, "correct": correct}
    print(f"{name}: {sum(correct)}/{len(correct)} correct")
    for level in levels:
        c = [ok for ok, i in zip(correct, idx) if test_levels[i] == level]
        print(f"   level {level}: {sum(c)}/{len(c)} ({100 * sum(c) / len(c):.0f}%)")

def by_operator(name):
    # level-1 accuracy for each alien word
    r = results[name]
    for op in OPS:
        c = [ok for ok, i in zip(r["correct"], r["idx"]) if test_levels[i] == 1 and test_problems[i].startswith(op)]
        print(f"   {op:5s} {sum(c)}/{len(c)}")

def show_examples(name, n=3):
    r = results[name]
    for j in range(0, len(r["idx"]), max(1, len(r["idx"]) // n))[:n]:
        i = r["idx"][j]
        print(f"{test_problems[i]}  (expected {test_answers[i]})\n   -> {r['outputs'][j][:160]!r}\n")

def show_attention(att, tokens):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(att.float().cpu(), cmap="Blues", vmin=0)
    ax.set_xticks(range(len(tokens)), tokens, rotation=90); ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel("attends to (key)"); ax.set_ylabel("token doing the looking (query)")
    plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

print(generate_response([{"role": "user", "content": "What is 12 + 30? End with: Final Answer: <integer>"}]))

---
## Part 1: Crack the code [10 points]
Here is part of the signal: nine solved problems, three for each alien word.

In [ ]:
for op in OPS:
    shown = [(p, a) for p, a, lv in zip(train_problems, train_answers, train_levels) if lv == 1 and p.startswith(op)][:3]
    print("     ".join(f"{p} = {a}" for p, a in shown))

Write down what each word means. `check_rules` uses your rules to solve all 300 training problems, nested ones
included, and tells you how many you got right.

In [ ]:
my_rules = {
    "vek":  lambda a, b: None,   # TODO
    "drom": lambda a, b: None,   # TODO
    "skal": lambda a, b: None,   # TODO
}

def solve_with(rules, problem):
    m = re.fullmatch(r"(\w+)\((.*)\)", problem.strip())
    if not m:
        return int(problem)
    op, inside, depth = m.group(1), m.group(2), 0
    for i, ch in enumerate(inside):
        depth += (ch == "(") - (ch == ")")
        if ch == "," and depth == 0:
            return rules[op](solve_with(rules, inside[:i]), solve_with(rules, inside[i + 1:]))

def check_rules(rules):
    ok = [solve_with(rules, p) == a for p, a in zip(train_problems, train_answers)]
    print(f"your rules solve {sum(ok)}/{len(ok)} training problems")

check_rules(my_rules)

**Question:** Which word was hardest to crack, and what gave it away? How many examples did you need?

**Your answer:**

---
## Part 2: Inside the transformer [15 points]

### 2a. Tokens [2 points]
A transformer never sees letters, only **tokens** from a fixed vocabulary.

In [ ]:
problem = "vek(drom(2, 3), skal(4, 9))"
# TODO: split the problem into tokens (tokenizer.tokenize)
tokens = None

print(tokens)
print(len(tokens), "tokens")

**Question:** How are the alien words tokenized? What does that suggest about whether the model has seen them before?

**Your answer:**

### 2b. Next-token probabilities [5 points]
The model scores every token in its vocabulary (the **logits**); softmax turns the scores into probabilities for the next token.

In [ ]:
def next_token_probs(text, k=5):
    ids = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()   # one score per vocabulary token
    # TODO: turn the logits into probabilities (softmax over the vocabulary)
    probs = None
    # TODO: the k most likely tokens and their probabilities (torch.topk)
    top = None
    return [(tokenizer.decode(int(i)), round(p.item(), 3)) for p, i in zip(top.values, top.indices)]

print("10 - 3 =      ", next_token_probs("10 - 3 = "))
print("skal(3, 10) = ", next_token_probs("skal(3, 10) = "))

**Question:** Both lines mean the same thing. How confident is the model about each, and is its top guess for `skal` right?

**Your answer:**

### 2c. Attention [6 points]
Run the model on `skal(3, 10) = ` and look at the **attention weights** in one layer, averaged over its heads.
Row *i* shows how much token *i* looks at every token (slides 13–21).

In [ ]:
text = "skal(3, 10) = "
ids = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    # TODO: run the model on ids with output_attentions=True; keep the result in `out`
    out = None

layer = 12
# TODO: out.attentions[layer] has shape (sentences, heads, tokens, tokens);
#       take our one sentence and average over the heads
att = None

show_attention(att, tokenizer.convert_ids_to_tokens(ids["input_ids"][0]))

**Question:** (1) Why is everything above the diagonal empty? (2) Which token gets the most attention? (3) Ignoring that token, what does the last token look at?

**Your answer:**

### 2d. Size [2 points]

In [ ]:
# TODO: total number of parameters (add up .numel() over model.parameters())
n_params = None

print(f"{n_params:,} parameters")

**Question:** The CNN from class (IC 5) had about 24,000 parameters. How many times bigger is this model?

**Your answer:**

---
## Part 3: Base prompting [5 points]
Ask for the answer with **no explanation** of the language: one user message, no system message.

In [ ]:
def build_base_prompt(problem):
    # TODO: return a list with one user message: [{"role": "user", "content": ...}]
    return None

print(build_base_prompt(test_problems[0]))

In [ ]:
evaluate("Base", build_base_prompt)
show_examples("Base")

---
## Part 4: System prompting [10 points]
Use a **system message** to define the three words and require `Final Answer: <integer>`. The user message holds the problem.

In [ ]:
def build_system_prompt(problem):
    system = (
        # TODO: define vek, drom and skal, and ask for "Final Answer: <integer>"
        ""
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Solve: {problem}"},
    ]

In [ ]:
evaluate("System", build_system_prompt)
by_operator("System")

**Question:** Look at the level-1 results by word. Which word does the model get wrong most, even though you defined it? Why?

**Your answer:**

---
## Part 5: Chain-of-thought [15 points]
Make the model **write its steps**: one operation per line, innermost first, translated to normal math, then
`Final Answer: <integer>`. Showing a worked example in the system message helps a lot.
Reference: [Wei et al., 2022](https://arxiv.org/abs/2201.11903).

In [ ]:
def build_cot_prompt(problem):
    system = (
        # TODO: define the words, ask for one step per line (innermost first, in normal math),
        # include one worked example, and finish with "Final Answer: <integer>"
        ""
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Solve: {problem}"},
    ]

▶ Steps take more tokens, so this run is slower *(about 4 minutes on a T4)*.

In [ ]:
evaluate("Chain-of-thought", build_cot_prompt, max_new_tokens=128)
by_operator("Chain-of-thought")
show_examples("Chain-of-thought")

**Question:** Compare levels 2 and 3 with Part 4. Why does writing the steps help on nested problems?

**Your answer:**

---
## Part 6: In-context learning [10 points]
Show the model **solved examples** from the training set, the way you cracked the code in Part 1: one per level,
as alternating user / assistant turns after your Part 4 system message. Never use test problems as examples.

In [ ]:
def get_icl_examples(n_per_level=1):
    # TODO: (problem, answer) pairs from the TRAINING set, n_per_level from each level
    return []

def build_icl_prompt(problem):
    messages = [{"role": "system", "content": "TODO: your system message from Part 4"}]
    for p, a in get_icl_examples():
        # TODO: a user turn with the example problem, then an assistant turn "Final Answer: <a>"
        pass
    messages.append({"role": "user", "content": f"Solve: {problem}"})
    return messages

for m in build_icl_prompt(test_problems[0]):
    print(f"{m['role']:>9}: {m['content'][:80]!r}")

In [ ]:
evaluate("In-context", build_icl_prompt)
by_operator("In-context")

**Question:** Three examples were enough for *you* in Part 1. Which words did they fix for the model, and which didn't they?

**Your answer:**

---
## Part 7: LoRA fine-tuning [25 points]
Prompting leaves the weights alone; **fine-tuning** changes them. LoRA ([Hu et al., 2021](https://arxiv.org/abs/2106.09685))
freezes the model and trains small low-rank matrices added to the attention layers. After training, the prompt has
**no definitions and no examples**: the language has to live in the weights.

### 7a. Configure LoRA [5 points]
`r` is the rank of the trainable matrices (try 4–16); `lora_alpha` is a scale, usually 2 × `r`;
`target_modules` picks the attention projections to adapt.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, TensorDataset

# TODO: a LoraConfig for causal language modeling (TaskType.CAUSAL_LM) with a rank r between 4 and 16,
#       lora_alpha = 2 * r, and target_modules for the query and value projections ("q_proj", "v_proj")
lora_config = None

▶ Load a fresh copy of the model and attach LoRA *(don't edit)*.

In [ ]:
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

**Question:** What percentage of the parameters will LoRA train?

**Your answer:**

### 7b. Train [20 points]
Each training example is the conversation `Solve: <problem>` → `Final Answer: <answer>`. The prompt tokens are
labeled `-100`, so only the answer counts toward the loss.

A causal language model predicts the **next** token: logit position *t* is scored against label *t + 1*, so drop the
last logit and the first label before the cross-entropy. The PyTorch pattern (adapt the names and shapes):
```python
scores = model(input_ids=x, attention_mask=mask).logits        # forward: (batch, positions, vocabulary)
loss = torch.nn.functional.cross_entropy(scores_2d, targets_1d, ignore_index=-100)
optimizer.zero_grad(); loss.backward(); optimizer.step()         # update
```

In [ ]:
def prepare_lora_data(problems, answers, max_length=96):
    ids_all, labels_all, attn_all = [], [], []
    for problem, answer in zip(problems, answers):
        prompt = [{"role": "user", "content": f"Solve: {problem}"}]
        full = prompt + [{"role": "assistant", "content": f"Final Answer: {answer}"}]
        prompt_len = len(tokenizer(tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True))["input_ids"])
        enc = tokenizer(tokenizer.apply_chat_template(full, tokenize=False), max_length=max_length,
                        truncation=True, padding="max_length", return_tensors="pt")
        ids, attn = enc["input_ids"][0], enc["attention_mask"][0]
        labels = ids.clone(); labels[:prompt_len] = -100; labels[attn == 0] = -100
        ids_all.append(ids); labels_all.append(labels); attn_all.append(attn)
    return torch.stack(ids_all), torch.stack(labels_all), torch.stack(attn_all)

train_ids, train_labels, train_attn = prepare_lora_data(train_problems, train_answers)
print("training tensors:", tuple(train_ids.shape))
print("the part the loss sees:", repr(tokenizer.decode(train_ids[0][train_labels[0] != -100])))

▶ Fill in the four TODOs *(about 4 minutes on a T4)*.

In [ ]:
loader = DataLoader(TensorDataset(train_ids, train_labels, train_attn), batch_size=4, shuffle=True)
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=2e-4)
lora_model.train()

for epoch in range(3):
    total = 0.0
    for input_ids, labels, attn in loader:
        input_ids, labels, attn = input_ids.to(device), labels.to(device), attn.to(device)

        # TODO 1: forward pass: run lora_model on input_ids and attention_mask, keep .logits
        logits = None

        # TODO 2: shift: drop the LAST logit position and the FIRST label
        shift_logits = None
        shift_labels = None

        # TODO 3: cross-entropy between shift_logits and shift_labels
        #         (flatten both, cast the logits to float, ignore_index=-100)
        loss = None

        # TODO 4: clear old gradients, backpropagate, take an optimizer step

        total += loss.item()
    print(f"epoch {epoch + 1}: average loss {total / len(loader):.3f}")

▶ Evaluate with the bare prompt `Solve: <problem>`.

In [ ]:
lora_model.eval()
evaluate("LoRA", lambda p: [{"role": "user", "content": f"Solve: {p}"}], model=lora_model)
by_operator("LoRA")

---
## Part 8: Analysis [10 points]

In [ ]:
names = [n for n in results if not n.startswith("Bonus")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
w = 0.8 / len(names)
for j, n in enumerate(names):
    r = results[n]
    lv = [100 * sum(ok for ok, i in zip(r["correct"], r["idx"]) if test_levels[i] == L) / 30 for L in (1, 2, 3)]
    ops = [100 * sum(ok for ok, i in zip(r["correct"], r["idx"]) if test_levels[i] == 1 and test_problems[i].startswith(o))
           / max(1, sum(test_levels[i] == 1 and test_problems[i].startswith(o) for i in r["idx"])) for o in OPS]
    for ax, vals in zip(axes, (lv, ops)):
        ax.bar_label(ax.bar([x + j * w for x in range(3)], vals, w, label=n), fmt="%.0f", fontsize=7)
for ax, labels, title in zip(axes, (["level 1", "level 2", "level 3"], list(OPS)), ("accuracy by level", "level 1: accuracy by word")):
    ax.set_xticks([x + w * (len(names) - 1) / 2 for x in range(3)], labels); ax.set_ylim(0, 110)
    ax.set_title(title); ax.set_ylabel("%")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

**1.** Which method is best at each level? Is one method best everywhere? [3 points]

**Your answer:**

**2.** Which methods learned that `skal` is reversed, and what do they have in common? [3 points]

**Your answer:**

**3.** Suggest one change that would raise level-3 accuracy, and say why it should work. [4 points]

**Your answer:**

---
## ★ Bonus: prompt leaderboard [+5 points]
Write the best prompt you can for **level 3**, using prompting only: no fine-tuning, no test problems inside the prompt.
The top three scores in the class earn the bonus. The reference chain-of-thought prompt scores about 20%.

In [ ]:
def my_best_prompt(problem):
    # TODO (optional): your best prompt
    return build_cot_prompt(problem)

evaluate("Bonus: my prompt", my_best_prompt, max_new_tokens=160, levels=(3,))